<a href="https://colab.research.google.com/github/garaimg/repositorios-git-pruebas/blob/main/colibri_que_es_si_funciona_y_como_probarlo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colibrí: qué es, si funciona de verdad y cómo probarlo en Colab

Proyecto: <https://github.com/JustVugg/colibri> · Licencia libre (Apache 2.0) · Autor: JustVugg

En este cuaderno vamos a hacer tres cosas:

1. **Explicar** qué es colibrí y cómo funciona, sin dar nada por sabido.
2. **Comprobar** si lo que promete el proyecto es verdad, con pruebas que cualquiera puede repetir.
3. **Usarlo de verdad**: pondremos a funcionar un modelo de inteligencia artificial que ocupa **más que toda la
   memoria de Colab** y le haremos preguntas.

⏱️ **Tarda unos 25–35 minutos en total**; casi todo es la descarga del modelo (23 GB). No hace falta tarjeta
gráfica: basta con el Colab gratuito normal.

## 1. El problema

Un modelo de lenguaje es, en el fondo, un archivo enorme lleno de números, sus **pesos**. Para que el
modelo escriba, el ordenador tiene que tener esos números a mano en la **memoria RAM**. La regla de siempre es
sencilla: **si el modelo no cabe en la memoria, no se puede usar**.

El problema es que los modelos buenos son enormes:

| Modelo | Tamaño (comprimido) |
|---|---:|
| Qwen3.6-35B | ~20 GB |
| DeepSeek V4 Flash | ~150 GB |
| GLM-5.2 | ~400 GB |

Un ordenador normal tiene entre 16 y 32 GB de memoria, así que, con la regla de siempre, estos modelos no se pueden
usar en casa.

## 2. La idea: estos modelos solo usan una parte pequeña cada vez

Muchos modelos modernos son **"modelos de expertos"**. Cada una de sus capas tiene dos partes:

- **Una parte común** que se usa siempre.
- **Muchos "expertos"**: pequeños trozos del modelo especializados en cosas distintas. GLM-5.2 tiene 256 por capa.
  Para cada palabra, un **selector** elige solo unos pocos (8 en GLM-5.2) y **el resto ni se toca**.

Los modelos escriben por **tokens**, trozos de palabra (una palabra suele ser uno o dos tokens). Para escribir cada
token, GLM-5.2 solo usa en torno al **5 % del modelo**.

## 3. Lo que hace colibrí

Colibrí aprovecha esa idea: **no carga el modelo entero en memoria**, sino que coloca cada parte donde le corresponde:

```
         más rápido ◄──────────────────────────────────────► más lento
  ┌─────────────────────┐   ┌───────────────────────┐   ┌───────────────────────┐
  │ Tarjeta gráfica     │   │   Memoria RAM         │   │   Disco               │
  │ (opcional)          │   │ la parte común        │   │ todos los expertos;   │
  │ los expertos que    │   │ + los expertos usados │   │ se leen solo cuando   │
  │ más se usan         │   │   hace poco           │   │ el selector los pide  │
  └─────────────────────┘   └───────────────────────┘   └───────────────────────┘
```

1. La **parte común** se carga en memoria al arrancar.
2. Los **expertos** se quedan en el disco. Cuando el selector pide uno, colibrí mira primero si lo tiene en memoria
   porque lo usó hace poco (a eso se le llama **caché**). Si no lo tiene, lo lee del disco. Cuando la caché se
   llena, quita el experto que lleva más tiempo sin usarse.
3. Mientras hace cuentas con unos expertos, **ya va leyendo los siguientes**, para no perder tiempo esperando.

**Lo importante:** que un experto esté en memoria o en el disco cambia **lo rápido** que va, porque la memoria es unas
100 veces más rápida que el disco, pero **no cambia lo que responde el modelo**. El proyecto lo presenta como una de
sus garantías, y aquí lo vamos a comprobar.

**Otros datos del proyecto:**

- Está escrito en **C**, un lenguaje de programación muy rápido, y no necesita instalar nada más.
- Guarda los números del modelo **comprimidos a 4 u 8 bits** (en vez de 16), para que ocupen menos.
- Se puede usar desde **cualquier programa preparado para ChatGPT o Claude**, cambiando solo la dirección a la que
  se conecta.

## 4. Qué promete el proyecto y cómo lo vamos a comprobar

| # | Lo que dice el proyecto | Cómo lo comprobamos |
|---|---|---|
| P1 | Está hecho en C, **no necesita instalar nada más** ni tarjeta gráfica | Lo compilamos y miramos qué necesita para funcionar |
| P2 | Responde **exactamente lo mismo** que la versión oficial del modelo | Comparamos, token a token, con la versión oficial de Hugging Face usando modelos de prueba en miniatura |
| P3 | Tener más o menos expertos en memoria cambia **la velocidad, no la respuesta** | Repetimos la prueba dejando en memoria 8, 2 y 1 expertos, y luego con el modelo de verdad |
| P4 | Se **instala con un solo comando** | Lo instalamos así |
| P5 | Funciona con modelos **más grandes que la memoria del ordenador** | Usamos un modelo de 23 GB en un Colab con unos 12 GB de memoria y medimos cuánta usa |
| P6 | Con ese modelo va a **6 tokens por segundo** en un ordenador con procesador de 8 núcleos | Medimos la velocidad en Colab, que tiene 2 núcleos |
| P7 | Se puede usar **como si fuera ChatGPT o Claude** desde otros programas | Lo ponemos en marcha como servidor y le hablamos de las dos formas |
| P8 | **GLM-5.2 (744.000 millones de números) funciona con unos 25 GB de memoria**, a 0,05–0,1 tokens por segundo | Hacemos las cuentas y medimos el disco de Colab (el modelo ocupa 429 GB y no cabe en Colab) |
| P9 | Sus cálculos internos son **exactos** | Prueba del propio proyecto: compara sus funciones rápidas con otras sencillas que sirven de referencia |

**Leyenda:** ✅ cierto · ⚠️ cierto, pero con matices · ❌ falso · ⏸️ no se puede comprobar aquí

## 5. Preparación

In [1]:
# ⚙️ Opciones
MODELO = "qwen36-35b"        # "qwen36-35b" (23 GB, conversación) o "qwen3-coder-30b" (19 GB, especializado en programar)
DIR_MODELOS = "/content/colibri-models"
REPO_DIR = "/content/colibri"
C_DIR = REPO_DIR + "/c"
USAR_MODELO_REAL = True      # False = solo las pruebas rápidas, sin descargar 23 GB (unos 2 minutos)

In [2]:
import os, re, json, subprocess, time, shutil
from IPython.display import display, Markdown
RESULTADOS = []

def ejecutar(orden, carpeta=None, lineas=12, entorno=None, limite=3600):
    # Ejecuta una orden de terminal, enseña sus últimas líneas y devuelve (código de salida, texto completo)
    p = subprocess.run(orden, shell=True, cwd=carpeta or C_DIR, env={**os.environ, **(entorno or {})}, timeout=limite,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors="replace")
    salida = p.stdout.rstrip().splitlines()
    for l in (salida[-lineas:] if lineas > 0 else []):
        print("   ", l)
    return p.returncode, p.stdout

def es(x, decimales=1):
    # Escribe un número a la española: coma para los decimales y punto para los miles
    if x is None:
        return "?"
    return f"{x:,.{decimales}f}".replace(",", "_").replace(".", ",").replace("_", ".")

def buscar(patron, texto, tipo=float):
    m = re.search(patron, texto)
    return tipo(m.group(1)) if m else None

def apuntar(n, promesa, resultado, explicacion):
    RESULTADOS.append({"#": n, "Lo que dice el proyecto": promesa, "Resultado": resultado, "Qué hemos visto": explicacion})
    display(Markdown(f"**{n} → {resultado}** · {explicacion}"))

ram_total = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
os.makedirs("/content", exist_ok=True)
disco_libre = shutil.disk_usage("/content").free / 2**30
print(f"Este Colab tiene: {os.cpu_count()} núcleos · {es(ram_total, 1)} GB de memoria · {es(disco_libre, 0)} GB libres en disco")

Este Colab tiene: 2 núcleos · 12,7 GB de memoria · 226 GB libres en disco


In [3]:
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/JustVugg/colibri {REPO_DIR}
!cd {REPO_DIR} && git log -1 --format="Versión que vamos a probar: %h, del %ad" --date=short

Versión que vamos a probar: 0a5504a1, del 2026-10-31


## 6. P1 · ¿Está hecho en C y no necesita nada más?

**Compilar** es convertir el código en un programa que el ordenador puede ejecutar. Compilamos dos versiones de
colibrí: la de los modelos GLM y la de Qwen3.6, que es la que usaremos luego con el modelo de verdad. Después
miramos qué piezas del sistema necesitan para funcionar.

In [4]:
t0 = time.time()
codigo, _ = ejecutar("make -j$(nproc) colibri qwen36 2>&1 | tail -1", lineas=1)
print(f"   → {'Compilado sin errores' if codigo == 0 else 'ERROR al compilar'} en {es(time.time()-t0, 0)} segundos")
piezas = set()
for programa in ("colibri", "qwen36"):
    _, salida = ejecutar(f"ldd ./{programa}", lineas=0)
    piezas |= set(re.findall(r"(lib[\w+-]+)\.so", salida))
extra = sorted(p for p in piezas if p not in ("libc", "libm", "libgomp", "libpthread", "libdl", "librt"))
print("Piezas del sistema que necesita:", ", ".join(sorted(piezas)), "(todas vienen de serie con el compilador)")
print("Piezas adicionales que haya que instalar:", extra or "ninguna")
familias = sorted(f for f in os.listdir(C_DIR) if f.endswith(".c") and f.split(".")[0] in
                  ("colibri", "glm53", "qwen36", "qwen38", "deepseek_v4", "deepseek_v41", "kimi_k3", "mimo", "inkling", "olmoe"))
print("Un archivo de código por cada familia de modelos:", ", ".join(familias))
apuntar("P1", "Hecho en C, sin instalar nada más ni tarjeta gráfica", "✅" if codigo == 0 and not extra else "❌",
        f"Se compila con lo que trae el sistema y no pide nada adicional; hay {len(familias)} versiones, una por familia de modelos")

    gcc -O3 -march=native -fopenmp -pthread -Wall -Wextra -Wno-unused-parameter -Wno-misleading-indentation -Wno-unused-function -MMD -MP qwen36.c      -o qwen36 -lm -fopenmp -pthread
   → Compilado sin errores en 93 segundos
Piezas del sistema que necesita: libc, libgomp, libm (todas vienen de serie con el compilador)
Piezas adicionales que haya que instalar: ninguna
Un archivo de código por cada familia de modelos: colibri.c, deepseek_v4.c, deepseek_v41.c, glm53.c, inkling.c, kimi_k3.c, mimo.c, olmoe.c, qwen36.c, qwen38.c


**P1 → ✅** · Se compila con lo que trae el sistema y no pide nada adicional; hay 10 versiones, una por familia de modelos

## 7. P2 y P3 · ¿Responde exactamente lo mismo que la versión oficial?

La prueba más exigente para un programa así es compararlo, **token a token**, con la versión oficial del modelo,
que es la de la librería `transformers` de Hugging Face.

Para no tener que descargar cientos de GB, el proyecto trae unos **modelos de prueba en miniatura**: tienen
exactamente la misma estructura que GLM-5.2 y que Qwen3.6, pero son diminutos y sus números son aleatorios. No sirven
para conversar (escriben cosas sin sentido), pero son perfectos para ver si colibrí hace las mismas cuentas que la
versión oficial.

Haremos dos comprobaciones:

- **Que escribe lo mismo**: colibrí y la versión oficial generan un texto, y comparamos token a token.
- **Que el resultado no depende de la memoria**: repetimos dejando en memoria **8, 2 y 1 expertos por capa**. Con 1,
  casi todos los expertos hay que leerlos del disco en cada token. Si colibrí está bien hecho, el texto será
  idéntico y solo cambiará cuántas veces encuentra el experto ya en memoria.

In [5]:
!pip install -q -U "transformers>=5.11" safetensors
print("Creando los modelos de prueba en miniatura...")
ejecutar("python3 tools/make_glm_oracle.py > /tmp/glm_mini.log 2>&1", lineas=0)
ejecutar("python3 tools/make_qwen36_tiny.py --out qwen36_tiny --ref-mode full --emit-ref qwen36_tiny/ref_full.json "
         "> /tmp/q36_mini.log 2>&1 && python3 tools/convert_qwen36.py --model qwen36_tiny --out qwen36_tiny_c --ebits 8 "
         ">> /tmp/q36_mini.log 2>&1", lineas=0)
print("GLM-5.2 en miniatura:", "listo" if os.path.exists(f"{C_DIR}/glm_tiny/model.safetensors") else "ERROR",
      "· Qwen3.6 en miniatura:", "listo" if os.path.exists(f"{C_DIR}/qwen36_tiny_c") else "ERROR")

Creando los modelos de prueba en miniatura...
GLM-5.2 en miniatura: listo · Qwen3.6 en miniatura: listo


In [6]:
filas, todo_igual = [], True
for expertos in (8, 2, 1):
    # GLM-5.2 en miniatura: escribe 20 tokens eligiendo siempre el más probable
    _, s = ejecutar(f"./colibri {expertos} 16 16", lineas=0,
                    entorno={"SNAP": "./glm_tiny", "COLI_TEMP": "0", "ORACLE_STRICT": "1", "CAP_RAISE": "0", "COLI_NO_OMP_TUNE": "1"})
    m = re.search(r"Matching tokens: (\d+)/(\d+)", s)
    filas.append(("GLM-5.2", expertos, f"{m[1]} de {m[2]}" if m else "?", buscar(r"hit rate: ([\d.]+)%", s)))
    todo_igual &= bool(m and m[1] == m[2])
    # Qwen3.6 en miniatura: 16 tokens
    _, s = ejecutar(f"./qwen36 {expertos} 8 qwen36_tiny/ref_full.json", lineas=0,
                    entorno={"COLI_DENSE_I8": "0", "SNAP": "qwen36_tiny_c"})
    m = re.search(r"Matching tokens: (\d+)/(\d+)", s)
    filas.append(("Qwen3.6", expertos, f"{m[1]} de {m[2]}" if m else "?", buscar(r"hit rate: ([\d.]+)%", s)))
    todo_igual &= bool(m and m[1] == m[2])

# Prueba extra con GLM: sobre un texto fijo, en cada una de sus 32 posiciones, ¿adivina el mismo siguiente token?
_, s = ejecutar("./colibri 64 16 16", lineas=0, entorno={"SNAP": "./glm_tiny", "COLI_TEMP": "0", "ORACLE_STRICT": "1",
                                                         "TF": "1", "ORACLE_TF_MAX_MISMATCHES": "2", "COLI_NO_OMP_TUNE": "1"})
pos = re.search(r"C vs oracle: (\d+)/(\d+)", s)

print(f"{'modelo':8} | {'expertos en memoria':>19} | {'tokens iguales a la oficial':>27} | veces que ya lo tenía en memoria")
for f in sorted(filas):
    print(f"{f[0]:8} | {f[1]:>19} | {f[2]:>27} | {es(f[3], 0)} %" if f[3] is not None else f)
print(f"\nGLM-5.2, posición a posición sobre un texto fijo: {pos[1]} de {pos[2]} iguales" if pos else "\nsin resultado")

modelo   | expertos en memoria | tokens iguales a la oficial | veces que ya lo tenía en memoria
GLM-5.2  |                   1 |                    20 de 20 | 36 %
GLM-5.2  |                   2 |                    20 de 20 | 44 %
GLM-5.2  |                   8 |                    20 de 20 | 88 %
Qwen3.6  |                   1 |                    16 de 16 | 6 %
Qwen3.6  |                   2 |                    16 de 16 | 22 %
Qwen3.6  |                   8 |                    16 de 16 | 80 %

GLM-5.2, posición a posición sobre un texto fijo: 32 de 32 iguales


In [7]:
apuntar("P2", "Responde exactamente lo mismo que la versión oficial",
        "✅" if todo_igual and pos and pos[1] == pos[2] else "⚠️",
        f"En los modelos en miniatura de GLM-5.2 y Qwen3.6, todos los tokens coinciden con la versión oficial "
        f"(y {pos[1]} de {pos[2]} posiciones en la prueba extra)" if pos else "sin resultado")
apuntar("P3", "La memoria cambia la velocidad, no la respuesta (modelos en miniatura)", "✅" if todo_igual else "❌",
        "Con 8, 2 o 1 expertos en memoria el texto es idéntico; lo único que cambia es cuántas veces hay que ir al disco")

**P2 → ✅** · En los modelos en miniatura de GLM-5.2 y Qwen3.6, todos los tokens coinciden con la versión oficial (y 32 de 32 posiciones en la prueba extra)

**P3 → ✅** · Con 8, 2 o 1 expertos en memoria el texto es idéntico; lo único que cambia es cuántas veces hay que ir al disco

## 8. Ahora en serio: un modelo de 35.000 millones de números en Colab

**Qwen3.6-35B-A3B** es un modelo de Alibaba con 35.000 millones de números, de los que solo usa unos 3.000 millones
por token. Comprimido para colibrí ocupa **23 GB, más que toda la memoria de Colab** (unos 12 GB). Con la regla de
siempre no se podría usar aquí. Es la misma situación que GLM-5.2 en un portátil, pero a escala de Colab.

### 8.1 P4 · Instalación con un solo comando

`coli setup` mira cómo es el ordenador, comprueba si el modelo cabe, prepara el programa y descarga el modelo. Si la
descarga se corta, continúa donde se quedó. Le decimos que no se quede funcionando al terminar, para poder hacer las
pruebas paso a paso.

In [8]:
RUTA_MODELO = f"{DIR_MODELOS}/{MODELO}"
if USAR_MODELO_REAL:
    t0 = time.time()
    codigo_inst, salida_inst = ejecutar(f"./coli setup --yes --model {MODELO} --backend cpu --no-start --no-browser "
                                        f"--dir {DIR_MODELOS}", lineas=25, limite=7200)
    minutos = (time.time() - t0) / 60
    if not os.path.exists(f"{RUTA_MODELO}/config.json"):
        # Plan B: descargar directamente desde Hugging Face
        print("\nLa instalación no dejó el modelo listo; lo descargamos directamente...")
        repo = re.search(r"\(([\w.-]+/[\w.-]+)\) ->", salida_inst)
        from huggingface_hub import snapshot_download
        snapshot_download(repo[1] if repo else "Kreuzzelg/qwen36-35b-a3b-colibri-i4-gs64", local_dir=RUTA_MODELO)
    tam_modelo = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(RUTA_MODELO) for f in fs) / 2**30
    print(f"\nEl modelo ocupa {es(tam_modelo, 1)} GB en disco; este Colab tiene {es(ram_total, 1)} GB de memoria. "
          f"Tiempo: {es(minutos, 1)} minutos")
    apuntar("P4", "Se instala con un solo comando", "✅" if codigo_inst == 0 else "⚠️",
            f"Un solo comando reconoció el ordenador, preparó el programa y descargó {es(tam_modelo, 1)} GB en {es(minutos, 1)} minutos"
            if codigo_inst == 0 else "La instalación no terminó sola; hubo que descargar el modelo a mano")
else:
    print("USAR_MODELO_REAL = False: nos saltamos la descarga")

      [........................]   0.0%  0.0 MB/23.0 GB  ?  ? left
      [##......................]  11.2%  2.6 GB/23.0 GB  84.1 MB/s  4 min left
      [#####...................]  21.4%  4.9 GB/23.0 GB  81.2 MB/s  4 min left
      [#######.................]  33.1%  7.6 GB/23.0 GB  84.0 MB/s  3 min left
      [##########..............]  43.1%  9.9 GB/23.0 GB  82.2 MB/s  3 min left
      [############............]  52.2%  12.0 GB/23.0 GB  79.6 MB/s  2 min left
      [##############..........]  62.2%  14.3 GB/23.0 GB  79.1 MB/s  2 min left
      [#################.......]  72.4%  16.7 GB/23.0 GB  79.0 MB/s  81 s left
      [###################.....]  82.8%  19.1 GB/23.0 GB  79.1 MB/s  50 s left
      [######################..]  95.4%  22.0 GB/23.0 GB  81.0 MB/s  13 s left
      [########################] 100.0%  23.0 GB  done
    
      expert profile: a starting history for Qwen3.6-35B-A3B (the experts its GPU tier loads first); your own use extends it
      downloading the prebuilt engi

**P4 → ✅** · Un solo comando reconoció el ordenador, preparó el programa y descargó 21,4 GB en 4,7 minutos

### 8.2 Primera pregunta

`coli run` arranca colibrí, le hace una pregunta y se cierra. Con `--no-think` le pedimos que conteste directamente,
sin escribir antes su razonamiento.

In [9]:
if USAR_MODELO_REAL:
    pregunta = "Explica en tres frases qué es un modelo de inteligencia artificial de expertos. Responde en español."
    t0 = time.time()
    ejecutar(f'./coli run --model {RUTA_MODELO} --no-think --ngen 160 "{pregunta}"', lineas=40, limite=1800)
    print(f"\n(tiempo total, incluido el arranque: {es(time.time()-t0, 0)} segundos)")

    [dense-i8] 291 matrices quantized during load, 7.2 GB f32 freed
    [qwen36] group-scaled experts: gs=64
    resident weights loaded in 36.3s | RSS after load: 2.93 GB
    [qwen36] expert format on disk: int4 packed (tier fmt=4)
    [qwen36] expert kernel: planar int4 in RAM (expert_ffn.h), QWEN_EXPERT_KERNEL=0 restores int8 unpack
    [qwen36] trunk: 291 int8 matrices interleaved by 16 for the CPU in 0.65s (Q36_ILV16=0 keeps the rows)
    [gateway] input modalities: text (the engine loaded no vision tower; a picture gets a 400)
    OpenAI-compatible API listening on http://127.0.0.1:49269/v1
    [api] 127.0.0.1 - "GET /health HTTP/1.1" 200 -
    [api] 127.0.0.1 - "GET /health HTTP/1.1" 200 -
    [api] 127.0.0.1 - "GET /health HTTP/1.1" 200 -
    [api] 127.0.0.1 - "GET /health HTTP/1.1" 200 -
    [api] 127.0.0.1 - "GET /health HTTP/1.1" 200 -
    [api] 127.0.0.1 - "GET /health HTTP/1.1" 200 -
    [api] 127.0.0.1 - "GET /health HTTP/1.1" 200 -
    [api] 127.0.0.1 - "GET /health HTTP

### 8.3 P5, P3 y P6 · Cuánta memoria usa, si cambia la respuesta y a qué velocidad va

Le hacemos la misma pregunta dos veces. Para que el resultado no dependa del azar, le pedimos que elija **siempre el
token más probable**. Cambiamos solo cuántos expertos puede guardar en memoria:

- **Pocos** (16 por capa): usa menos memoria, pero tiene que leer más del disco.
- **Más** (48 por capa): usa más memoria, pero tiene que ir menos al disco.

En cada caso medimos **la memoria máxima que ocupa el programa**, **cuántas veces encuentra el experto ya en memoria**,
**la velocidad** y **el texto que escribe**. Si colibrí cumple lo que promete, la memoria usada quedará **muy por debajo
de los 23 GB** que ocupa el modelo y **el texto será idéntico** las dos veces.

> Nota: el sistema también guarda en memoria trozos de los archivos que ha leído, para ir más rápido si los vuelve a
> necesitar. Esa memoria no cuenta aquí y el sistema la libera solo cuando hace falta.

In [10]:
def preguntar_directo(expertos, pregunta, n_tokens=48):
    with open("/tmp/pregunta.txt", "w") as f:
        f.write(f"<|im_start|>user\n{pregunta}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n")
    t0 = time.time()
    # El texto escrito sale por un canal y las estadísticas (memoria, caché, velocidad) por otro: los leemos por separado
    p = subprocess.run(f"./qwen36 {expertos} 4 /tmp/pregunta.txt", shell=True, cwd=C_DIR, timeout=3600, text=True,
                       errors="replace", capture_output=True,
                       env={**os.environ, "SNAP": RUTA_MODELO, "N_NEW": str(n_tokens), "COLI_TEMP": "0",
                            "NOSTREAM": "1", "CAP_RAISE": "0"})
    s = p.stderr
    return {"expertos": expertos, "memoria": buscar(r"PEAK RSS: ([\d.]+) GB", s), "aciertos": buscar(r"hit rate: ([\d.]+)%", s),
            "velocidad": buscar(r"Speed: ([\d.]+) tok/s", s), "arranque": buscar(r"loaded in ([\d.]+)s", s),
            "texto": p.stdout.strip() or None, "segundos": time.time() - t0, "registro": s}

if USAR_MODELO_REAL:
    pregunta = "¿Cuál es la capital de Francia y por qué es famosa? Responde en dos frases."
    pruebas = [preguntar_directo(n, pregunta) for n in (16, 48)]
    for r in pruebas:
        print(f"── Con {r['expertos']} expertos por capa en memoria ──")
        if r["memoria"] is None:
            print(r["registro"][-1500:])
            continue
        print(f"   Memoria máxima usada: {es(r['memoria'], 2)} GB · encontró el experto en memoria el {es(r['aciertos'], 0)} % "
              f"de las veces · velocidad: {es(r['velocidad'], 2)} tokens por segundo")
        print(f"   Texto: {r['texto']}\n")

── Con 16 expertos por capa en memoria ──
   Memoria máxima usada: 3,04 GB · encontró el experto en memoria el 58 % de las veces · velocidad: 0,40 tokens por segundo
   Texto: La capital de Francia es París, conocida mundialmente por sus monumentos icónicos como la Torre Eiffel y el Museo del Louvre. Además, es un centro global de arte, moda y gastronomía que atrae a millones de turistas

── Con 48 expertos por capa en memoria ──
   Memoria máxima usada: 5,15 GB · encontró el experto en memoria el 76 % de las veces · velocidad: 0,50 tokens por segundo
   Texto: La capital de Francia es París, conocida mundialmente por sus monumentos icónicos como la Torre Eiffel y el Museo del Louvre. Además, es un centro global de arte, moda y gastronomía que atrae a millones de turistas



In [11]:
if USAR_MODELO_REAL and all(r["memoria"] for r in pruebas):
    mem_max = max(r["memoria"] for r in pruebas)
    apuntar("P5", "Funciona con modelos más grandes que la memoria", "✅" if mem_max < tam_modelo and mem_max < ram_total else "⚠️",
            f"Modelo de {es(tam_modelo, 1)} GB en un Colab de {es(ram_total, 1)} GB: el programa usó como máximo "
            f"{es(pruebas[0]['memoria'], 1)} GB (16 expertos) y {es(pruebas[1]['memoria'], 1)} GB (48 expertos)")
    iguales = pruebas[0]["texto"] is not None and pruebas[0]["texto"] == pruebas[1]["texto"]
    apuntar("P3b", "La memoria cambia la velocidad, no la respuesta (modelo de verdad)", "✅" if iguales else "⚠️",
            f"El texto {'es idéntico' if iguales else 'NO es igual'} en las dos pruebas; encontró el experto en memoria el "
            f"{es(pruebas[0]['aciertos'], 0)} % y el {es(pruebas[1]['aciertos'], 0)} % de las veces")
    mejor = max(r["velocidad"] for r in pruebas)
    apuntar("P6", "6 tokens por segundo en un ordenador de 8 núcleos", "⚠️",
            f"En Colab, con {os.cpu_count()} núcleos, ha ido a {es(mejor, 2)} tokens por segundo ({es(mejor/os.cpu_count(), 2)} por "
            f"núcleo, frente a {es(6.0/8, 2)} por núcleo del dato publicado). La velocidad depende mucho del ordenador")
elif USAR_MODELO_REAL:
    apuntar("P5", "Funciona con modelos más grandes que la memoria", "❌", "El programa no terminó: mira el mensaje de arriba")

**P5 → ✅** · Modelo de 21,4 GB en un Colab de 12,7 GB: el programa usó como máximo 3,0 GB (16 expertos) y 5,2 GB (48 expertos)

**P3b → ✅** · El texto es idéntico en las dos pruebas; encontró el experto en memoria el 58 % y el 76 % de las veces

**P6 → ⚠️** · En Colab, con 2 núcleos, ha ido a 0,50 tokens por segundo (0,25 por núcleo, frente a 0,75 por núcleo del dato publicado). La velocidad depende mucho del ordenador

### 8.4 P7 · ¿Se puede usar como si fuera ChatGPT o Claude?

Muchos programas y aplicaciones están preparados para hablar con ChatGPT (OpenAI) o con Claude (Anthropic) a través de
internet. `coli serve` pone colibrí a funcionar como un **servidor** que entiende esos mismos "idiomas", así que esos
programas pueden usar el modelo de tu ordenador sin cambiar nada más que la dirección. Le mandamos una pregunta de
cada forma.

In [12]:
import urllib.request

def pedir(ruta, datos=None, cabeceras=None, limite=900):
    req = urllib.request.Request("http://127.0.0.1:8000" + ruta, data=json.dumps(datos).encode() if datos else None,
                                 headers={"Content-Type": "application/json", **(cabeceras or {})})
    with urllib.request.urlopen(req, timeout=limite) as r:
        return json.load(r)

if USAR_MODELO_REAL:
    servidor = subprocess.Popen(f"./coli serve --model {RUTA_MODELO} --port 8000", shell=True, cwd=C_DIR,
                                stdout=open("/tmp/servidor.log", "w"), stderr=subprocess.STDOUT)
    print("Arrancando el servidor", end="")
    for _ in range(600):
        try:
            modelos = pedir("/v1/models", limite=5); break
        except Exception:
            print(".", end="", flush=True); time.sleep(2)
    else:
        modelos = None
    print("\nModelos disponibles:", [m["id"] for m in modelos["data"]] if modelos
          else "el servidor no responde (mira /tmp/servidor.log)")

Arrancando el servidor.....
Modelos disponibles: ['qwen3.6-colibri']


In [13]:
ok_chatgpt = ok_claude = False
if USAR_MODELO_REAL and modelos:
    nombre = modelos["data"][0]["id"]
    # Como lo haría un programa preparado para ChatGPT
    try:
        t0 = time.time()
        r = pedir("/v1/chat/completions", {"model": nombre, "max_tokens": 80, "enable_thinking": False, "temperature": 0,
                  "messages": [{"role": "user", "content": "Di 'hola' en cinco idiomas, en una sola línea."}]})
        texto = r["choices"][0]["message"].get("content")
        ok_chatgpt = bool(texto)
        print(f"Al estilo ChatGPT ({es(time.time()-t0, 0)} s): {texto}")
    except Exception as e:
        print("Al estilo ChatGPT: error", e)
    # Como lo haría un programa preparado para Claude
    try:
        t0 = time.time()
        r = pedir("/v1/messages", {"model": nombre, "max_tokens": 80, "thinking": {"type": "disabled"},
                  "messages": [{"role": "user", "content": "¿Cuánto es 17 por 23? Responde solo el número."}]},
                  cabeceras={"anthropic-version": "2023-06-01"})
        trozos = [b.get("text") for b in r.get("content", []) if b.get("type") == "text"]
        ok_claude = r.get("type") == "message" and any(trozos)
        print(f"Al estilo Claude ({es(time.time()-t0, 0)} s): {' '.join(filter(None, trozos))}")
    except Exception as e:
        print("Al estilo Claude: error", e)
if USAR_MODELO_REAL:
    ejecutar("./coli stop", lineas=1)
    servidor.terminate()
    apuntar("P7", "Se puede usar como si fuera ChatGPT o Claude", "✅" if ok_chatgpt and ok_claude else "⚠️",
            f"Respondió {'bien' if ok_chatgpt else 'MAL'} a la petición al estilo ChatGPT y "
            f"{'bien' if ok_claude else 'MAL'} a la petición al estilo Claude")

Al estilo ChatGPT (99 s): Hola (español), Hello (inglés), Bonjour (francés), Ciao (italiano), Olá (portugués).
Al estilo Claude (58 s): 391
      ✓ stopped — RAM released


**P7 → ✅** · Respondió bien a la petición al estilo ChatGPT y bien a la petición al estilo Claude

## 9. P8 · La gran promesa: GLM-5.2 con unos 25 GB de memoria

GLM-5.2 no cabe en Colab: hay que descargar 429 GB y Colab tiene unos 100 GB de disco. Pero sí podemos comprobar dos
cosas:

1. **Que las cuentas cuadran**, usando los datos oficiales del modelo: cuánto ocupa cada experto, cuántos hay y
   cuánto hay que leer del disco por cada token.
2. **Que la velocidad que anuncian es creíble.** Si cada token necesita leer unos 11 GB de expertos y casi ninguno
   está ya en memoria, la velocidad máxima es más o menos **la velocidad del disco dividida entre 11 GB**.

In [14]:
datos, origen = None, "datos oficiales del modelo (config.json de Hugging Face)"
try:
    with urllib.request.urlopen("https://huggingface.co/zai-org/GLM-5.2-FP8/resolve/main/config.json", timeout=20) as r:
        datos = json.load(r)
except Exception as e:
    print("No se pudieron descargar los datos oficiales:", e)
    datos = {"hidden_size": 6144, "num_hidden_layers": 78, "first_k_dense_replace": 3, "n_routed_experts": 256,
             "num_experts_per_tok": 8, "moe_intermediate_size": 2048, "num_nextn_predict_layers": 1}
    origen = "datos que aparecen en el propio proyecto"
d = lambda k: int(datos.get(k) or 0)
ancho, capas, por_capa, elegidos = d("hidden_size"), d("num_hidden_layers"), d("n_routed_experts"), d("num_experts_per_tok")
capas_comunes, tam_experto, extra = d("first_k_dense_replace"), d("moe_intermediate_size"), d("num_nextn_predict_layers")
numeros_experto = 3 * ancho * tam_experto
mb_experto = numeros_experto * 0.5 / 1e6                # 4 bits = medio byte por número
capas_expertos = capas - capas_comunes
total_expertos = (capas_expertos + extra) * por_capa
gb_por_token = capas_expertos * elegidos * mb_experto / 1e3
print(f"Fuente: {origen}")
print(f"Cada experto tiene {es(numeros_experto/1e6, 1)} millones de números → {es(mb_experto, 1)} MB comprimido a 4 bits")
print(f"Hay {es(total_expertos, 0)} expertos → {es(total_expertos*mb_experto/1e3, 0)} GB, que se quedan en el disco")
print(f"Cada token usa {capas_expertos} capas × {elegidos} expertos = {capas_expertos*elegidos} expertos → "
      f"{es(gb_por_token, 1)} GB que hay que leer si ninguno está ya en memoria")

Fuente: datos oficiales del modelo (config.json de Hugging Face)
Cada experto tiene 37,7 millones de números → 18,9 MB comprimido a 4 bits
Hay 19.456 expertos → 367 GB, que se quedan en el disco
Cada token usa 75 capas × 8 expertos = 600 expertos → 11,3 GB que hay que leer si ninguno está ya en memoria


In [15]:
print("Midiendo la velocidad del disco de Colab (leemos trozos de 19 MB, el tamaño de un experto, en sitios al azar)...")
!head -c 2G /dev/urandom > /content/prueba_disco.bin && sync
_, s = ejecutar("gcc -O2 -fopenmp iobench.c -o /tmp/iobench && /tmp/iobench /content/prueba_disco.bin 19 60 4 1", lineas=0)
os.remove("/content/prueba_disco.bin")
disco = buscar(r"-> ([\d.]+) GB/s", s)
print(f"Disco de Colab: {es(disco, 2)} GB por segundo\n")
print(f"Velocidad máxima esperada para GLM-5.2 = velocidad del disco ÷ {es(gb_por_token, 1)} GB por token:")
for quien, gbs, ya_en_memoria, anunciado in [("Portátil del autor (disco de ~1 GB/s)", 1.0, 0, "0,05–0,1"),
                                             ("Mac con M5 Max (14,2 GB/s, 23 % ya en memoria)", 14.2, 0.23, "1,06"),
                                             (f"Este Colab ({es(disco, 2)} GB/s)", disco, 0, "—")]:
    if gbs:
        print(f"  {quien:47s} → hasta {es(gbs/(gb_por_token*(1-ya_en_memoria)), 2)} tokens/s   (anunciado: {anunciado})")
apuntar("P8", "GLM-5.2 funciona con unos 25 GB de memoria", "⚠️",
        f"Las cuentas cuadran: unos 10 GB en memoria y {es(total_expertos*mb_experto/1e3, 0)} GB de expertos en el disco. "
        f"La velocidad anunciada es creíble (hasta {es(1.0/gb_por_token, 2)} tokens/s con un disco normal), pero eso son "
        "10–20 segundos por palabra. No se puede probar en Colab porque el modelo ocupa 429 GB")

Midiendo la velocidad del disco de Colab (leemos trozos de 19 MB, el tamaño de un experto, en sitios al azar)...
Disco de Colab: 0,74 GB por segundo

Velocidad máxima esperada para GLM-5.2 = velocidad del disco ÷ 11,3 GB por token:
  Portátil del autor (disco de ~1 GB/s)           → hasta 0,09 tokens/s   (anunciado: 0,05–0,1)
  Mac con M5 Max (14,2 GB/s, 23 % ya en memoria)  → hasta 1,63 tokens/s   (anunciado: 1,06)
  Este Colab (0,74 GB/s)                          → hasta 0,07 tokens/s   (anunciado: —)


**P8 → ⚠️** · Las cuentas cuadran: unos 10 GB en memoria y 367 GB de expertos en el disco. La velocidad anunciada es creíble (hasta 0,09 tokens/s con un disco normal), pero eso son 10–20 segundos por palabra. No se puede probar en Colab porque el modelo ocupa 429 GB

## 10. P9 · ¿Sus cálculos internos son exactos?

Para ir rápido, colibrí usa instrucciones especiales del procesador que hacen muchas multiplicaciones a la vez. Esas
funciones rápidas deben dar **exactamente** el mismo resultado que otras sencillas que hacen la cuenta paso a paso, y
el proyecto trae una prueba que lo comprueba en miles de casos:

In [16]:
!grep -m1 "model name" /proc/cpuinfo
codigo, s = ejecutar("gcc -O3 -march=native -fopenmp -I. tests/test_int_kernel_exact.c -o /tmp/exacto -lm && /tmp/exacto", lineas=1)
n = re.search(r"(\d+) checks, (\d+) failures", s)
apuntar("P9", "Sus cálculos internos son exactos", "✅" if codigo == 0 and n and n[2] == "0" else "❌",
        f"{es(int(n[1]), 0)} comprobaciones y {n[2]} errores" if n else "sin resultado")

model name	: Intel(R) Xeon(R) CPU @ 2.20GHz
    int-kernel exactness: 11312 checks, 0 failures (PASS / avx2)


**P9 → ✅** · 11.312 comprobaciones y 0 errores

## 11. Resumen

In [17]:
import pandas as pd
tabla = pd.DataFrame(RESULTADOS)
cuenta = {k: (tabla["Resultado"] == k).sum() for k in ["✅", "⚠️", "❌", "⏸️"]}
display(Markdown(f"✅ {cuenta['✅']} ciertas · ⚠️ {cuenta['⚠️']} ciertas con matices · ❌ {cuenta['❌']} falsas · "
                 f"⏸️ {cuenta['⏸️']} sin comprobar"))
display(tabla.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))

✅ 8 ciertas · ⚠️ 2 ciertas con matices · ❌ 0 falsas · ⏸️ 0 sin comprobar

#,Lo que dice el proyecto,Resultado,Qué hemos visto
P1,"Hecho en C, sin instalar nada más ni tarjeta gráfica",✅,"Se compila con lo que trae el sistema y no pide nada adicional; hay 10 versiones, una por familia de modelos"
P2,Responde exactamente lo mismo que la versión oficial,✅,"En los modelos en miniatura de GLM-5.2 y Qwen3.6, todos los tokens coinciden con la versión oficial (y 32 de 32 posiciones en la prueba extra)"
P3,"La memoria cambia la velocidad, no la respuesta (modelos en miniatura)",✅,"Con 8, 2 o 1 expertos en memoria el texto es idéntico; lo único que cambia es cuántas veces hay que ir al disco"
P4,Se instala con un solo comando,✅,"Un solo comando reconoció el ordenador, preparó el programa y descargó 21,4 GB en 4,7 minutos"
P5,Funciona con modelos más grandes que la memoria,✅,"Modelo de 21,4 GB en un Colab de 12,7 GB: el programa usó como máximo 3,0 GB (16 expertos) y 5,2 GB (48 expertos)"
P3b,"La memoria cambia la velocidad, no la respuesta (modelo de verdad)",✅,El texto es idéntico en las dos pruebas; encontró el experto en memoria el 58 % y el 76 % de las veces
P6,6 tokens por segundo en un ordenador de 8 núcleos,⚠️,"En Colab, con 2 núcleos, ha ido a 0,50 tokens por segundo (0,25 por núcleo, frente a 0,75 por núcleo del dato publicado). La velocidad depende mucho del ordenador"
P7,Se puede usar como si fuera ChatGPT o Claude,✅,Respondió bien a la petición al estilo ChatGPT y bien a la petición al estilo Claude
P8,GLM-5.2 funciona con unos 25 GB de memoria,⚠️,"Las cuentas cuadran: unos 10 GB en memoria y 367 GB de expertos en el disco. La velocidad anunciada es creíble (hasta 0,09 tokens/s con un disco normal), pero eso son 10–20 segundos por palabra. No se puede probar en Colab porque el modelo ocupa 429 GB"
P9,Sus cálculos internos son exactos,✅,11.312 comprobaciones y 0 errores


## 12. Conclusiones

**¿Es verdad lo que dice colibrí?** En lo importante, sí:

- **La idea funciona.** Un modelo de 23 GB funciona en un Colab con unos 12 GB de memoria, y el programa ni siquiera
  necesita toda esa memoria, porque los expertos se leen del disco solo cuando hacen falta.
- **No se inventa nada.** En los modelos de prueba, colibrí escribe exactamente lo mismo que la versión oficial, token a
  token, y la respuesta no cambia según cuánta memoria le dejes: solo cambia la velocidad.
- **Es fácil de usar.** Se instala con un comando y otros programas pueden usarlo como si fuera ChatGPT o Claude.

**El precio es la velocidad.** Leer del disco es mucho más lento que leer de la memoria:

- Con modelos medianos, como el que hemos probado, se puede usar en un ordenador normal.
- Con GLM-5.2, "funciona con 25 GB" quiere decir **unos 10–20 segundos por palabra** en un portátil normal. Es una
  demostración técnica muy interesante, pero no algo para usar a diario. Con más memoria y un disco rápido mejora
  bastante: en torno a 1–2 tokens por segundo en un Mac con M5 Max o en un ordenador con 128 GB.

**En pocas palabras:** con colibrí la pregunta deja de ser "¿me cabe el modelo?" y pasa a ser "¿cuánto estoy dispuesto
a esperar?", y lo consigue sin cambiar lo que responde el modelo.